### Problem 001: Implement Trie (Prefix Tree) (LeetCode 208)

### Problem Definition and Constraints
A prefix tree (also known as a trie) is a tree data structure used to efficiently store and retrieve keys in a set of strings. 
Implement the `PrefixTree` (or `Trie`) class:
* `Trie()` Initializes the prefix tree object.
* `void insert(String word)` Inserts the string `word` into the prefix tree.
* `boolean search(String word)` Returns `true` if the string `word` is in the prefix tree (i.e., was inserted before), and `false` otherwise.
* `boolean startsWith(String prefix)` Returns `true` if there is a previously inserted string `word` that has the prefix `prefix`, and `false` otherwise.

**Examples:**
* **Example 1:**
  * **Input:** `["Trie", "insert", "dog", "search", "dog", "search", "do", "startsWith", "do", "insert", "do", "search", "do"]`
  * **Output:** `[null, null, true, false, true, null, true]`

* Constraints:
  * 1 <= word.length, prefix.length <= 1000
  * `word` and `prefix` are made up of lowercase English letters.
  * At most $3 \cdot 10^4$ calls in total will be made to `insert`, `search`, and `startsWith`.

### Trie Approach (Node and Path Traversal)
We build the Trie using a helper `TrieNode` class. The main `Trie` class simply holds a pointer to the empty `root` node.
1. **Insert:** We start at the root and look at each letter in the word. If the current letter isn't in the current node's children, we create a new `TrieNode`. Then we step into that child. When the loop finishes, we mark the final node with `is_word = True`.
2. **Search:** We walk down the path letter by letter. If we ever look for a letter that doesn't exist, we return `False`. If we make it to the end of the word, we return the node's `is_word` flag (because finding the path "d-o" doesn't mean "do" is a word if we only inserted "dog").
3. **StartsWith:** This is exactly the same as `search`, but if we successfully reach the end of the prefix path, we immediately return `True` (we don't care if `is_word` is True or False).

* **Time Complexity:** $O(L)$ for each function call — Where $L$ is the length of the `word` or `prefix`. We do a single loop through the characters of the string, doing an $O(1)$ dictionary lookup at each step.
* **Space Complexity:** $O(N \cdot L)$ — Where $N$ is the total number of words inserted, and $L$ is the average length of those words. In the worst case (no shared prefixes), we create a new node for every single letter of every single word. Space for `search` and `startsWith` is $O(1)$.

In [1]:
class TrieNode:
    def __init__(self):
        # Dictionary mapping a character to its child TrieNode
        # Example: {"a": TrieNode(), "b": TrieNode()}
        self.children = {}
        
        # Flag to mark the end of a valid inserted word
        self.is_word = False

class PrefixTree:
    def __init__(self):
        # The root node represents the empty string.
        # All words start branching out from here.
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        # Always start at the top of the tree
        curr = self.root
        
        for char in word:
            # If the path doesn't exist yet, create it
            if char not in curr.children:
                curr.children[char] = TrieNode()
            # Step down into the child node
            curr = curr.children[char]
            
        # We finished placing the last letter. Mark this exact node as a word.
        curr.is_word = True

    def search(self, word: str) -> bool:
        curr = self.root
        
        for char in word:
            # If the path breaks before the word ends, the word isn't here
            if char not in curr.children:
                return False
            # Step down into the child node
            curr = curr.children[char]
            
        # We found the path! But is it a complete word, or just a prefix?
        # If we inserted "apple", searching "app" will land on a node where is_word == False.
        return curr.is_word

    def startsWith(self, prefix: str) -> bool:
        curr = self.root
        
        for char in prefix:
            # If the path breaks, the prefix doesn't exist
            if char not in curr.children:
                return False
            curr = curr.children[char]
            
        # We survived the loop, which means the exact prefix path exists.
        # We don't care if it's a full word or not, so just return True.
        return True

### Problem 002: Design Add and Search Word Data Structure (LeetCode 211)

### Problem Definition and Constraints
Design a data structure that supports adding new words and searching for existing words.
Implement the `WordDictionary` class:
* `void addWord(word)` Adds `word` to the data structure.
* `bool search(word)` Returns `true` if there is any string in the data structure that matches `word` or `false` otherwise. `word` may contain dots `.` where dots can be matched with any letter.

**Examples:**
* **Example 1:**
  * **Input:** `["WordDictionary","addWord","addWord","addWord","search","search","search","search"]` <br> `[[],["day"],["bay"],["may"],["say"],["day"],[".ay"],["b.."]]`
  * **Output:** `[null, null, null, null, false, true, true, true]`

* Constraints:
  * 1 <= word.length <= 20
  * `word` in `addWord` consists of lowercase English letters.
  * `word` in `search` consist of `.` or lowercase English letters.
  * There will be at most 2 dots in `word` for search queries.
  * At most 10,000 calls will be made to `addWord` and `search`.

### Trie + DFS (Backtracking) Approach
We use the exact same `TrieNode` building block as before. 
1. **addWord:** Literally identical to `insert` from the standard Trie. No changes.
2. **search:** Because of the `.` wildcard, a simple `for` loop won't work anymore. We need to use a recursive `dfs` function. 
   * The `dfs` takes our current `index` in the word and the `node` we are currently standing on.
   * If the current character is a standard letter, we step into that child node normally.
   * If the current character is a `.`, we loop through *all* existing children of the current node and recursively call `dfs` on every single one. If any of those recursive paths return `True`, the whole search returns `True`.

* **Time Complexity:** 
  * `addWord`: $O(L)$ — Where $L$ is the length of the word.
  * `search`: $O(L)$ for normal words without dots. If there are dots, it is technically $O(26^D \cdot L)$ where $D$ is the number of dots. However, since the constraints guarantee at most 2 dots, $26^2$ is a constant (676), so the time scales linearly with word length.
* **Space Complexity:** $O(N \cdot L)$ for storing all the words in the Trie, plus $O(L)$ memory for the recursive `dfs` call stack during the search.

In [2]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False

class WordDictionary:
    def __init__(self):
        self.root = TrieNode()

    def addWord(self, word: str) -> None:
        curr = self.root
        for char in word:
            if char not in curr.children:
                curr.children[char] = TrieNode()
            curr = curr.children[char]
        curr.is_word = True

    def search(self, word: str) -> bool:
        # We need a DFS helper function to handle the '.' wildcard branching
        def dfs(index, node):
            curr = node
            
            for i in range(index, len(word)):
                char = word[i]
                
                if char == ".":
                    # WILDCARD: We don't know which path to take!
                    # So, we iterate through EVERY existing child of the current node.
                    for child in curr.children.values():
                        # If ANY of the paths find the rest of the word, return True
                        if dfs(i + 1, child):
                            return True
                    
                    # If we tried all paths and none worked out, this is a dead end
                    return False
                
                else:
                    # NORMAL CHARACTER: Just do a standard Trie traversal
                    if char not in curr.children:
                        return False
                    curr = curr.children[char]
            
            # We reached the end of the word. Is it a valid stopping point?
            return curr.is_word
            
        # Start the recursive search at index 0, standing at the root node
        return dfs(0, self.root)

### Problem 003: Word Search II (LeetCode 212) [HARD]

### Problem Definition and Constraints
Given a 2-D grid of characters `board` and a list of strings `words`, return all words that are present in the grid.
For a word to be present, it must be formed by connecting horizontally or vertically neighboring cells. The same cell may not be used more than once in a single word path.

**Examples:**
* **Example 1:**
  * **Input:** `board = [["a","b"], ["c","d"]]`, `words = ["abcb", "ba", "a", "c", "ab"]`
  * **Output:** `["ba", "a", "c", "ab"]`
  * **Explanation:** "abcb" is invalid because it tries to reuse the "b" cell twice in one path.
* **Example 2:**
  * **Input:** `board = [["x","o"], ["x","o"]]`, `words = ["xoxo"]`
  * **Output:** `[]`

* Constraints:
  * 1 <= board.length, board[i].length <= 12
  * 1 <= words.length <= 30,000
  * 1 <= words[i].length <= 10

### Core Logic: The "Reverse Search" (Trie + 2D Backtracking)
If we run our standard "Word Search I" algorithm for every single word in the list, we would launch a grid search 30,000 times. That will instantly trigger a Time Limit Exceeded (TLE) error.

Instead, we reverse the paradigm: **We search the board exactly once, looking for all words simultaneously.**
We do this by inserting all 30,000 words into a single Trie. 
We then launch a DFS from every cell on the board. As we step into a new cell, we step down the Trie. 
*   **Massive Pruning:** If the character we step on does not exist as a child in our current Trie node, we instantly abort the DFS. This single check might simultaneously rule out 5,000 words that started with the wrong prefix!
*   **Duplicate Prevention:** Instead of a simple `is_word` boolean, we store the actual string at the terminal Trie node (`node.word = "cat"`). When our DFS lands on a node that has a word, we append it to our results and instantly set `node.word = None`. This ensures we never accidentally add the same word twice if it appears in multiple places on the board.

### Approach: Trie-Optimized DFS
1. **Build the Trie:** Create a `TrieNode` class. Iterate through the `words` list and insert every word. At the final node of each word, store the word itself.
2. **The Grid DFS Engine:** Track the current row `r`, column `c`, and the `node` we are currently standing on in the Trie.
   * **Base Cases / Filter:** If `(r, c)` is out of bounds, already in our `path` set, or `board[r][c]` is not a child of the current Trie `node`, return immediately.
   * **Choose:** Add `(r, c)` to `path`. Step the `node` forward to `node.children[board[r][c]]`.
   * **Record:** If `node.word` exists, we found a match! Add it to our results list, and cross it off the Trie by setting `node.word = None`.
   * **Explore:** Recursively call DFS in all 4 directions (Up, Down, Left, Right).
   * **Undo:** Remove `(r, c)` from `path` so other branches can use it.
3. **The Starter Loop:** Iterate over every single cell in the board and launch the DFS passing in the root of the Trie.

* **Time Complexity:** $O(m \cdot n \cdot 4 \cdot 3^{L-1} + s)$ — Where $s$ is the total characters across all words (to build the Trie), and $L$ is the maximum length of a word. From each of the $m \cdot n$ cells, we branch in 4 directions, then 3 directions (since we can't go backward), up to depth $L$.
* **Space Complexity:** $O(s)$ — The Trie stores all the words. The DFS call stack and path set take $O(L)$ space.

In [3]:
from typing import List

class TrieNode:
    def __init__(self):
        self.children = {}
        # Instead of a boolean, we store the actual word here at the end of the path.
        # This makes it O(1) to append the word to our results without passing a string builder down the DFS.
        self.word = None

class Solution:
    def findWords(self, board: List[List[str]], words: List[str]) -> List[str]:
        # 1. BUILD THE TRIE
        root = TrieNode()
        for w in words:
            curr = root
            for char in w:
                if char not in curr.children:
                    curr.children[char] = TrieNode()
                curr = curr.children[char]
            # Store the complete word at the terminal node
            curr.word = w
            
        ROWS, COLS = len(board), len(board[0])
        res = []
        path = set()
        
        # 2. THE DFS ENGINE
        def dfs(r, c, node):
            # Out of bounds check
            if r < 0 or c < 0 or r >= ROWS or c >= COLS:
                return
                
            # Cycle detection (snake biting its own tail)
            if (r, c) in path:
                return
                
            char = board[r][c]
            
            # Massive Pruning: If this character isn't a valid next step in the Trie, abort!
            if char not in node.children:
                return
                
            # Step down the Trie
            curr_node = node.children[char]
            
            # Did we just finish spelling a word?
            if curr_node.word:
                res.append(curr_node.word)
                # Erase the word from the Trie so we don't find it a second time
                curr_node.word = None
                
            # Mark cell as visited for this DFS branch
            path.add((r, c))
            
            # Explore all 4 directions
            dfs(r + 1, c, curr_node)
            dfs(r - 1, c, curr_node)
            dfs(r, c + 1, curr_node)
            dfs(r, c - 1, curr_node)
            
            # Backtrack: unmark the cell so parallel paths can use it
            path.remove((r, c))

        # 3. THE STARTER LOOP
        # Launch the DFS from every single cell on the board
        for r in range(ROWS):
            for c in range(COLS):
                dfs(r, c, root)
                
        return res


        """
        =========================================================
        SIMULATION NOTES: 
        board = [["o", "a", "a", "n"], ["e", "t", "a", "e"]]
        words = ["oath", "pea", "eat", "rain"]
        =========================================================
        
        Trie Structure:
        root -> 'o' -> 'a' -> 't' -> 'h' (word="oath")
             -> 'p' -> 'e' -> 'a' (word="pea")
             -> 'e' -> 'a' -> 't' (word="eat")
             -> 'r' -> 'a' -> 'i' -> 'n' (word="rain")
             
        DFS Starts at (0,0) -> 'o':
          Is 'o' in root.children? YES.
          curr_node = root.children['o']
          
          Explore Down (1,0) -> 'e':
            Is 'e' in curr_node.children? NO! (The only child of 'o' is 'a')
            Path instantly pruned! (Saved massive time)
            
          Explore Right (0,1) -> 'a':
            Is 'a' in curr_node.children? YES.
            curr_node = curr_node.children['a']
            
            Explore Down (1,1) -> 't':
              Is 't' in curr_node.children? YES.
              
              Explore Down ... fails bounds.
              Explore Left ... fails bounds.
              Explore Right ... fails bounds.
              
              ... DFS finds 'h' eventually, res.append("oath"), curr_node.word = None.
              
        The Trie inherently blocks any DFS path that isn't actively spelling a 
        word that exists in the list!
        """